## Brain Tumor Detection

### [Model Training using Tensorflow]


#### Reading the dataset

In [1]:
import os

train_dir = "../data/brain-tumor-mri-dataset/Training"
test_dir = "../data/brain-tumor-mri-dataset/Testing"

# To load and shuffle the training data
train_paths = []
train_labels = []

for label in os.listdir(train_dir):
    label_dir = os.path.join(train_dir, label)
    for img_name in os.listdir(label_dir):
        train_paths.append(os.path.join(label_dir, img_name))
        train_labels.append(label)

print(f"Number of training samples: {len(train_paths)}")
print(f"Unique labels: {set(train_labels)}")

Number of training samples: 5600
Unique labels: {'notumor', 'glioma', 'pituitary', 'meningioma'}


#### Shuffling the training data

In [2]:
from sklearn.utils import shuffle

for label in os.listdir(test_dir):
    label_dir = os.path.join(test_dir, label)
    print(f"Total test samples for label '{label}': {len(os.listdir(label_dir))}")
    for img_name in os.listdir(label_dir):
        print(f"Test image: {os.path.join(label_dir, img_name)} with label: {label}")
        train_paths.append(os.path.join(label_dir, img_name))
        train_labels.append(label)


train_paths, train_labels = shuffle(train_paths, train_labels, random_state=42)

Total test samples for label 'glioma': 400
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_1.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_10.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_100.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_101.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_102.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_103.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_104.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_105.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_106.jpg with label: glioma
Test image: ../data/brain-tumor-mri-dataset/Testing/glioma/Te-gl_107.jpg with label: glioma
Test image: ../data/brain-tumor-mri-data

#### Image Preprocessing (Helper Functions)


In [3]:
import random
import numpy as np
from PIL import Image, ImageEnhance
from tensorflow.keras.preprocessing.image import load_img

I0000 00:00:1777886732.065115  241735 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777886732.125157  241735 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777886733.773637  241735 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


1. Image Augmentation function


In [4]:
def augment_image(image):
    image = Image.fromarray(np.uint8(image))
    image = ImageEnhance.Brightness(image).enhance(
        random.uniform(0.8, 1.2)
    )  # Random brightness
    image = ImageEnhance.Contrast(image).enhance(
        random.uniform(0.8, 1.2)
    )  # Random contrast
    image = np.array(image) / 255.0  # Normalize pixel values to [0, 1]
    return image

2. Load images and apply augmentation


In [5]:
def open_images(paths):
    images = []
    for path in paths:
        image = load_img(path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
        image = augment_image(image)
        images.append(image)
    return np.array(images)

3. Encoding labels (convert label names to integers)


In [6]:
def encode_label(labels):
    unique_labels = os.listdir(train_dir)  # Ensure unique labels are determined
    encoded = [unique_labels.index(label) for label in labels]
    return np.array(encoded)

4. Data generator for batching


In [7]:
def datagen(paths, labels, batch_size=12, epochs=1):
    for _ in range(epochs):
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i : i + batch_size]
            batch_images = open_images(batch_paths)  # Open and augment images
            batch_labels = labels[i : i + batch_size]
            batch_labels = encode_label(batch_labels)  # Encode labels
            yield batch_images, batch_labels  # Yield the batch

#### Defining our training loop
[Selecting VGG16 as our Pre-trained model and using Transfer Learning to perform hyperparameter finetuning]

In [8]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16

1. Defining trainable parameters for VGG16

In [9]:
IMAGE_SIZE = 128
num_classes = len(os.listdir(train_dir))

# Base Model
base_model = VGG16(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights="imagenet"
)

# Freeze all layers first
for layer in base_model.layers:
    layer.trainable = False

# Unfreeze the last few layers (Conv block 5)
for layer in base_model.layers[-4:]:
    layer.trainable = True

W0000 00:00:1777886734.366075  241735 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


2. Defining our final neural network

In [10]:
from tensorflow.keras.layers import Input, Flatten, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

In [11]:
model = Sequential(
    [
        Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)),
        base_model,
        Flatten(),
        Dropout(0.3),
        Dense(128, activation="relu"),
        Dropout(0.2),   # Dropping 20% of the neurons to prevent overfitting
        Dense(num_classes, activation="softmax"),
    ]
)

# Compile
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["sparse_categorical_accuracy"],
)

3. Training our final neural network

In [12]:
# Train
batch_size = 20
epochs = 5
steps = int(len(train_paths) / batch_size)

# datagen is a generator yielding (x_batch, y_batch)
train_dataset = datagen(train_paths, train_labels, batch_size=batch_size, epochs=epochs)

history = model.fit(train_dataset, epochs=epochs, steps_per_epoch=steps)

Epoch 1/5


I0000 00:00:1777886735.220646  241735 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


360/360 ━━━━━━━━━━━━━━━━━━━━ 103s 281ms/step - loss: 0.4591 - sparse_categorical_accuracy: 0.8275
Epoch 2/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 101s 282ms/step - loss: 0.2334 - sparse_categorical_accuracy: 0.9132
Epoch 3/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 101s 280ms/step - loss: 0.1534 - sparse_categorical_accuracy: 0.9436
Epoch 4/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 105s 293ms/step - loss: 0.1086 - sparse_categorical_accuracy: 0.9625
Epoch 5/5
360/360 ━━━━━━━━━━━━━━━━━━━━ 108s 300ms/step - loss: 0.0773 - sparse_categorical_accuracy: 0.9718


4. Saving our trained model

In [13]:
final_accuracy = history.history['sparse_categorical_accuracy'][-1]
final_loss = history.history['loss'][-1]

In [14]:
save_directory = "../models"

if not os.path.exists(save_directory):
    os.makedirs(save_directory)
    print(f"Created directory: {save_directory}")

model_name = f"brain_tumor_vgg16_acc_{final_accuracy:.4f}_loss_{final_loss:.4f}.h5"
full_save_path = os.path.join(save_directory, model_name)

model.save(full_save_path)
print(f"Model saved to: {full_save_path}")

Model saved to: ../models/brain_tumor_vgg16_acc_0.9718_loss_0.0773.h5
